# TrendLens — 05 · Cluster Interpretation (Phase 5)

Stage 8: representative images (Phase 3) → **BLIP-1 captioning** (CPU-friendly; BLIP-2/API pluggable) → deterministic aggregation → cluster name, description, characteristics, confidence.

> **Integrity:** names/descriptions/characteristics are VLM-derived **INTERPRETATIONS, not ground truth**. Low confidence means low caption agreement — reported honestly.

In [1]:
import sys
from pathlib import Path
REPO = Path.cwd()
if not (REPO / "config.py").exists():
    for p in Path.cwd().parents:
        if (p / "config.py").exists():
            REPO = p; break
sys.path.insert(0, str(REPO))

import json, numpy as np, pandas as pd
import config
from src import interpretation

## 1 · Load representative images (Phase 3 artifacts)

In [2]:
reps = {int(k): v for k, v in json.loads(
    (config.CLUSTER_METADATA_DIR / "representatives.json").read_text()).items()}
print("clusters:", len(reps))
print("reps per cluster:", sorted({len(v) for v in reps.values()}))

clusters: 29
reps per cluster: [9]


## 2 · Load BLIP captioning model (cached locally, runs offline)

> NOTE: if you re-run the notebook, keep `use_cached=true` below to reuse the captions already computed in the driver run instead of re-captioning on CPU.

In [3]:
use_cached = True
cached = config.CLUSTER_METADATA_DIR / "cluster_captions.json"

if use_cached and cached.exists():
    caps_by_cluster = {
        i["cluster_id"]: i["sample_captions"]
        for i in json.loads(cached.read_text())["interpretations"]
    }
    print("reused cached captions for", len(caps_by_cluster), "clusters")
else:
    model, processor, device = interpretation.load_blip()
    print("BLIP on", device)
    caps_by_cluster = interpretation.caption_representatives(
        model, processor, reps, k=4, device=device)
    print("captioned", len(caps_by_cluster), "clusters")

reused cached captions for 29 clusters


In [4]:
sample = pd.DataFrame([
    {"cluster": c, "i": i, "caption": cap}
    for c, caps in sorted(caps_by_cluster.items())
    for i, cap in enumerate(caps)
])
sample.head(8)

,cluster,i,caption
0,0,0,a dog laying on a green blanket
1,0,1,a small dog laying on a bed with a blanket
2,0,2,a dog laying on the ground with its tongue out
3,0,3,a dog laying on a couch with its paws on the c...
4,1,0,a cat with a white face
5,1,1,a cat with yellow eyes
6,1,2,a cat laying on the floor
7,1,3,a cat with its mouth open


## 3 · Aggregate captions into an interpretation per cluster

In [5]:
interpreted = interpretation.interpret_all_clusters(caps_by_cluster)
df = pd.DataFrame([{k: v for k, v in i.items() if k != "sample_captions"} for i in interpreted])
df.sort_values("confidence", ascending=False).head(10)

,cluster_id,name,description,characteristics,confidence
12,12,leo leo,A visual cluster whose images are described by...,"[leo, jumping, doing, handstant, tennis, court]",0.4488
8,8,scr scr,A visual cluster whose images are described by...,"[scr, white, motorcycle, yellow, red, car]",0.3165
15,15,graffiti wall,A visual cluster whose images are described by...,"[graffiti, wall, hole, green, yellow, covered]",0.1520
26,26,close eye,a close up of a person ' s eye,"[eye, close, blue, tooth, brush, mouth]",0.1018
3,3,moon seen,A visual cluster whose images are described by...,"[moon, seen, sky, nasa, visible, dark]",0.0895
1,1,cat white,A visual cluster whose images are described by...,"[cat, white, face, yellow, eyes, laying]",0.0794
0,0,dog laying,A visual cluster whose images are described by...,"[dog, laying, blanket, couch, green, small]",0.0792
21,21,building clock,A visual cluster whose images are described by...,"[building, windows, balks, clock, side, large]",0.0742
4,4,fish swimming,A visual cluster whose images are described by...,"[fish, tank, swimming, blue, inside, purple]",0.0693
13,13,skateboard doing,a man on a skateboard doing a trick,"[skateboard, doing, trick, skateboarder, ridin...",0.0693


In [6]:
df["confidence"].describe()

count    29.000000
mean      0.073255
std       0.093006
min       0.000000
25%       0.033100
50%       0.045100
75%       0.074200
max       0.448800
Name: confidence, dtype: float64

## 4 · Persist + human-readable report

In [7]:
path = interpretation.save_interpretations(interpreted)
rep = interpretation.write_report(interpreted)
print(path.name, "|", rep.name)

cluster_captions.json | captions_report.md


## Phase 5 checkpoint
- [x] Representative images → BLIP captions (CPU: ~6s/cluster)
- [x] Deterministic aggregation → name / description / characteristics / confidence
- [x] Pluggable VLM (BLIP-2 on GPU or API model = drop-in)
- [x] Honest labelling: interpretations ≠ ground truth; low-confidence clusters flagged

**Next (Phase 6):** FAISS vector store + RAG-style retrieval — embed cluster names/descriptions (Stage 10) and build retrieval/evaluation (Stage 12).